In [ ]:
import os
import pandas as pd
import seaborn as sns
%matplotlib widget
from matplotlib import pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
USE_MS = True

In [ ]:
# log_dirs = ["accuracy_data/", "accuracy_data3","accuracy_data2"]
log_dirs = ["data/srsran/fast"]
log_files = []
for log_dir in log_dirs:
    log_files += [f"{log_dir}/{i}" for i in os.listdir(log_dir) if (not i.startswith(".") and i.endswith(".txt"))]
log_files

In [ ]:
data = {}
for path in log_files:
    file_data = []
    filename = path.rsplit('.', 1)[0].rsplit('/', 1)[1].split("_")
    application = filename[0]
    rate = float(filename[1])
    if USE_MS:
        rate *= 1000.0
    with open(path, 'r') as f:
        lines = f.readlines()
    prev = None
    for line in lines:
        line = line.strip()
        if ':' not in line:
            continue
        ts = line.split(' ')[0].split(':')
        if len(ts) < 3:
            continue
        hour = float(ts[0])
        minute = float(ts[1])
        second = float(ts[2])
        second += 60*minute
        second += 60*60*hour
        ms = second*1000.0
        if prev is not None:
            if USE_MS:
                diff = ms - prev
            else:
                diff = second - prev
            file_data.append(diff)
        prev = ms
    if application not in data:
        data[application] = {}
    data[application][rate] = file_data



In [ ]:
# df = pd.DataFrame(
#     [
#         {"expected": expected, "measured": measured, "error": measured-expected}
#         for expected, measurements in data.items()
#         for measured in measurements
#     ]
# )
df = pd.DataFrame(
    [
        {"Application": application, "Expected": expected, "Measured": measured, "Error": measured-expected}
        for application, intervals in data.items()
        for expected, measurements in intervals.items()
        for measured in measurements
    ]
)
df["Absolute Error"] = abs(df["Error"])

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.lineplot(
    ax=ax,
    data=df,
    x="Expected",
    y="Measured",
    hue="Application",
    hue_order=["ping", "pping"],
    style="Application",
    style_order=["ping", "pping"],
    markers=True,
    dashes=False,
    palette="colorblind"
)
xmin = df["Expected"].min()
xmax = df["Expected"].max()
ax.plot([xmin, xmax], [xmin, xmax], "k--", label="y=x", alpha=0.5)
ax.set_xlabel("Expected Interval (ms)")
ax.set_ylabel("Measured Interval (ms)")
ax.xaxis.set_major_locator(MultipleLocator(5))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.lineplot(
    ax=ax,
    data=df,
    x="Expected",
    y="Error",
    hue="Application",
    hue_order=["ping", "pping"],
    style="Application",
    style_order=["ping", "pping"],
    markers=True,
    dashes=False,
    palette="colorblind"
)
ax.set_xlabel("Expected Interval (ms)")
ax.set_ylabel("Error (ms)")
ax.xaxis.set_major_locator(MultipleLocator(50))

In [ ]:
stats = df.groupby("Application")["Absolute Error"].agg(["min", "mean", "max"])
print("Absolute Error Statistics (ms):")
stats

In [ ]:
percent_reduction = (stats.loc["ping"] - stats.loc["pping"]) / stats.loc["ping"] * 100
print("Percent Absolute Error Reduction:")
percent_reduction